In [1]:
import sparknlp
from pyspark.sql import SparkSession
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner, Stemmer
from pyspark.ml import Pipeline

##### Read Data

In [2]:
spark = sparknlp.start(memory="8g")
# Task 2.6
ratings_df = spark.read.csv("hdfs://localhost:9000/home/zhiyang/data/Books_rating.csv", header=True, inferSchema=True, multiLine=True)
ratings_df = ratings_df.withColumnRenamed("review/text", "review_text")
ratings_df.printSchema()

26/03/27 20:42:25 WARN Utils: Your hostname, Zhiyangs-Macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.86.77 instead (on interface en0)
26/03/27 20:42:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/zhiyang/.ivy2/cache
The jars for the packages stored in: /Users/zhiyang/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e05d5c7e-94a6-4951-96aa-7b5f4a59ac8e;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/zhiyang/anaconda3/envs/dbms/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in local-m2-cache
	found org.apache.httpcomponents#httpclient;4.5.13 in local-m2-cache
	found org.apache.httpcomponents#httpcore;4.4.13 in local-m2-cache
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review_text: string (nullable = true)



##### Base Pipline

In [3]:
# Convert text → document
document_assembler = DocumentAssembler() \
    .setInputCol("review_text") \
    .setOutputCol("document")

# Tokenize
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# Normalize (lowercase, remove punctuation)
normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(True)

# Remove stopwords
stopwords_cleaner = StopWordsCleaner() \
    .setInputCols(["normalized"]) \
    .setOutputCol("clean_tokens") \
    .setCaseSensitive(False)

# Stem words
stemmer = Stemmer() \
    .setInputCols(["clean_tokens"]) \
    .setOutputCol("stem")

# Build pipeline
pipeline_base = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stopwords_cleaner,
    stemmer
])

model_base = pipeline_base.fit(ratings_df)
processed_rating_df = model_base.transform(ratings_df)
processed_rating_df.select("document", "token", "normalized", "clean_tokens", "stem").show(1, truncate=True)

+--------------------+--------------------+--------------------+--------------------+--------------------+
|            document|               token|          normalized|        clean_tokens|                stem|
+--------------------+--------------------+--------------------+--------------------+--------------------+
|[{document, 0, 45...|[{token, 0, 3, Th...|[{token, 0, 3, th...|[{token, 17, 21, ...|[{token, 17, 21, ...|
+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 1 row



##### Task 2.7 <br> We select:<br> (1). sentiment analysis,<br>(2). Language Detector <br>(3). named entity recognition

##### (1). sentiment analysis

In [4]:
from sparknlp.annotator import ViveknSentimentModel
from sparknlp.annotator import PerceptronModel, NerCrfModel, NerConverter, WordEmbeddingsModel
# (1). sentiment analysis
sentiment_stage = ViveknSentimentModel.pretrained() \
    .setInputCols(["document", "token"]) \
    .setOutputCol("sentiment")
pipline_senti = Pipeline(stages=[document_assembler, tokenizer, sentiment_stage])
model_senti = pipline_senti.fit(ratings_df)
processed_rating_df_senti = model_senti.transform(ratings_df)
processed_rating_df_senti.select("review_text", "sentiment.result").show(truncate=True)

sentiment_vivekn download started this may take some time.
Approximate size to download 873.6 KB
[ | ]sentiment_vivekn download started this may take some time.
Approximate size to download 873.6 KB


26/03/27 20:42:53 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:42:53 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Download done! Loading the resource.
[OK!]
+--------------------+----------+
|         review_text|    result|
+--------------------+----------+
|This is only for ...|[negative]|
|I don't care much...|[negative]|
|"If people become...|[positive]|
|Theodore Seuss Ge...|[negative]|
|"Philip Nel - Dr....|[negative]|
|"""Dr. Seuss: Ame...|[negative]|
|Theodor Seuss Gie...|[negative]|
|"When I recieved ...|[positive]|
|"Trams (or any pu...|[positive]|
|As far as I am aw...|[positive]|
|I just finished t...|[positive]|
|"Many small churc...|[negative]|
|I just finished r...|[negative]|
|"I hadn't been a ...|[negative]|
|I bought this boo...|[positive]|
|"I have to admit,...|[negative]|
|"This is a self-p...|[positive]|
|When I first read...|[positive]|
|I read the review...|[positive]|
|"I really enjoyed...|[positive]|
+--------------------+----------+
only showing top 20 rows



##### (2). Part-of-Speech Tagging

In [5]:
posTagger = PerceptronModel.pretrained("pos_anc", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("pos")
pipline_lan_detect = Pipeline(stages=[document_assembler, tokenizer, posTagger])
model_lan_tag = pipline_lan_detect.fit(ratings_df)
processed_rating_df_tag = model_lan_tag.transform(ratings_df)
processed_rating_df_tag.select("review_text", "pos.result").show(truncate=True)

pos_anc download started this may take some time.


26/03/27 20:42:58 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 3.9 MB
[ | ]

26/03/27 20:42:59 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:42:59 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


pos_anc download started this may take some time.
Approximate size to download 3.9 MB
Download done! Loading the resource.


[ / ]

[OK!]
+--------------------+--------------------+
|         review_text|              result|
+--------------------+--------------------+
|This is only for ...|[DT, VBZ, RB, IN,...|
|I don't care much...|[PRP, VBP, VB, RB...|
|"If people become...|['', IN, NNS, VBP...|
|Theodore Seuss Ge...|[NNP, NNP, NNP, (...|
|"Philip Nel - Dr....|['', NNP, NNP, :,...|
|"""Dr. Seuss: Ame...|[NN, NNP, ., NNP,...|
|Theodor Seuss Gie...|[NNP, NNP, NNP, V...|
|"When I recieved ...|['', WRB, PRP, VB...|
|"Trams (or any pu...|['', NNP, (, CC, ...|
|As far as I am aw...|[RB, RB, IN, PRP,...|
|I just finished t...|[PRP, RB, VBD, DT...|
|"Many small churc...|['', JJ, JJ, NNS,...|
|I just finished r...|[PRP, RB, VBD, VB...|
|"I hadn't been a ...|['', PRP, VBP, VB...|
|I bought this boo...|[PRP, VBD, DT, NN...|
|"I have to admit,...|['', PRP, VBP, TO...|
|"This is a self-p...|['', DT, VBZ, DT,...|
|When I first read...|[WRB, PRP, JJ, VB...|
|I read the review...|[PRP, VBP, DT, NN...|
|"I really enjoyed...|['',

##### (3). named entity recognition

In [6]:
posTagger = PerceptronModel.pretrained("pos_anc", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("pos")

embeddings = WordEmbeddingsModel.pretrained("glove_100d", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings")

ner_crf = NerCrfModel.pretrained("ner_crf", "en") \
    .setInputCols(["document", "token", "pos", "embeddings"]) \
    .setOutputCol("ner")

ner_converter = NerConverter() \
    .setInputCols(["document", "token", "ner"]) \
    .setOutputCol("entities")

pipline_entity = Pipeline(stages=[document_assembler, tokenizer, posTagger, embeddings, ner_crf, ner_converter])
model_entity = pipline_entity.fit(ratings_df)
processed_rating_df_entity = model_entity.transform(ratings_df)
processed_rating_df_entity.select("review_text", "entities.result").show(truncate=True)

pos_anc download started this may take some time.
Approximate size to download 3.9 MB
[ | ]

26/03/27 20:43:04 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


[OK!]
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[ | ]

26/03/27 20:43:07 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:43:07 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:43:07 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


glove_100d download started this may take some time.
Approximate size to download 145.3 MB
Download done! Loading the resource.
[OK!]
ner_crf download started this may take some time.
Approximate size to download 10.2 MB
[ | ]

26/03/27 20:43:10 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:43:10 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/27 20:43:10 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_crf download started this may take some time.
Approximate size to download 10.2 MB
Download done! Loading the resource.


[ / ]

[OK!]


+--------------------+--------------------+
|         review_text|              result|
+--------------------+--------------------+
|This is only for ...|[Julie Strain, It...|
|I don't care much...|[Dr, Seuss, Phili...|
|"If people become...|                  []|
|Theodore Seuss Ge...|[Theodore Seuss G...|
|"Philip Nel - Dr....|[Philip Nel, Dr, ...|
|"""Dr. Seuss: Ame...|[Seuss, American ...|
|Theodor Seuss Gie...|[Theodor Seuss Gi...|
|"When I recieved ...|[Christmas, Oh, S...|
|"Trams (or any pu...|             [Trams]|
|As far as I am aw...|[Dr, Seuss, Phili...|
|I just finished t...|[David Ray, &quot...|
|"Many small churc...|                  []|
|I just finished r...|   [IS, David Ray's]|
|"I hadn't been a ...|[David R, Ray, An...|
|I bought this boo...|[Unfortunately, G...|
|"I have to admit,...|         [Traveling]|
|"This is a self-p...|[Ms, Haddon's, Am...|
|When I first read...|            [Haddon]|
|I read the review...|             [WHOLE]|
|"I really enjoyed...|          

##### Task 2.8 <br> We select <br> 1. identifying frequent named entities (authors, publishers, etc.) <br> 2. analyzing sentiment distribution across reviews

##### 1. analyzing sentiment distribution across reviews

In [7]:
from pyspark.sql.functions import col
ratings_df = ratings_df.repartition(20) # repartition for parallel
model_senti = pipline_senti.fit(ratings_df)
processed_rating_df_senti = model_senti.transform(ratings_df)

# 1. Access the first element of the array directly
# This turns the 'Array of Structs' into a simple 'String' column
skinny_df = processed_rating_df_senti.select(
    col("sentiment")[0]["result"].alias("sentiment_label")
)

# 2. Run stats
skinny_df.groupBy("sentiment_label").count().orderBy("sentiment_label").show()

+---------------+-------+
|sentiment_label|  count|
+---------------+-------+
|           null|     41|
|             na|   7094|
|       negative|1363002|
|       positive|1629669|
+---------------+-------+



##### 2. analyzing the correlation between sentiment and price

In [8]:
from pyspark.sql.functions import col, avg
ratings_df = ratings_df.repartition(20) # repartition for parallel
model_senti = pipline_senti.fit(ratings_df)
processed_rating_df_senti = model_senti.transform(ratings_df)

# 1. Access the first element of the array directly
# This turns the 'Array of Structs' into a simple 'String' column
skinny_df = processed_rating_df_senti.select(
    col("price").alias("price"),
    col("sentiment")[0]["result"].alias("sentiment_label")
)

# 2. Run stats
skinny_df.groupBy("sentiment_label") \
    .agg(avg("price").alias("avg_price")) \
    .orderBy("sentiment_label") \
    .show()

+---------------+------------------+
|sentiment_label|         avg_price|
+---------------+------------------+
|           null|            10.885|
|             na|20.163124470787476|
|       negative| 22.07519479499713|
|       positive| 21.52229912326797|
+---------------+------------------+

